# Objective

This notebook introduces the first asynchronous, multi-threaded stage of the profiling pipeline.

The goal is to decouple event production from event processing using a thread-safe bounded queue.

In the previous notebook, events were processed synchronously:

```text
SyntheticEventSource
        ↓
   ProfileEvent
        ↓
Serializer / Sink
```

In this notebook, the pipeline becomes:

```text
Producer Thread
      ↓
BoundedQueue<ProfileEvent>
      ↓
Consumer Thread
      ↓
Serializer / Sink
```

This design allows event generation and event processing to run independently.

The notebook focuses on the following objectives:

- Implement a reusable `BoundedQueue<T>` template.
- Protect shared queue state using `std::mutex`.
- Coordinate producers and consumers using `std::condition_variable`.
- Prevent unbounded memory growth by limiting queue capacity.
- Run event production and consumption on separate threads.
- Use `std::jthread` and `std::stop_token` for modern C++20 thread management.
- Implement a clean shutdown mechanism so blocked threads can exit safely.
- Validate the pipeline under different producer and consumer speeds.

---

## Why a Queue Is Needed

In a real profiling system, event production and event processing may run at different speeds.

For example:

```text
GPU / Profiler
   produces events quickly
          ↓
        Queue
          ↓
Network / Storage
   processes events more slowly
```

Without an intermediate queue, a slow consumer could directly block the event producer.

A queue provides temporary buffering and allows both sides to operate asynchronously.

---

## Why the Queue Is Bounded

An unbounded queue can continue growing if the producer is faster than the consumer.

For example:

```text
Producer: 1,000,000 events/s
Consumer:   300,000 events/s
```

The queue would continuously accumulate events:

```text
queue size
   ↑
   ↑
   ↑
   ↑
memory exhaustion
```

A bounded queue introduces a fixed capacity:

```text
Producer
   ↓
┌───────────────────┐
│   BoundedQueue    │
│ capacity = N      │
└───────────────────┘
   ↓
Consumer
```

When the queue becomes full, the producer must wait until space becomes available.

This creates basic backpressure and prevents uncontrolled memory growth.

---

## Threading Model

The initial implementation uses two threads:

```text
Producer Thread
    |
    | next_event()
    v
BoundedQueue<ProfileEvent>
    |
    | pop()
    v
Consumer Thread
    |
    v
ConsoleSink / Serializer
```

The producer is responsible for:

```text
read/generate event
        ↓
push event into queue
```

The consumer is responsible for:

```text
wait for event
        ↓
remove event from queue
        ↓
process event
```

---

## Synchronization

Because the producer and consumer access the same queue concurrently, the shared state must be protected.

The implementation will use:

```text
std::mutex
std::condition_variable
```

The mutex protects access to the queue and its state.

Condition variables allow threads to sleep efficiently while waiting for a condition.

The producer waits when:

```text
queue is full
```

The consumer waits when:

```text
queue is empty
```

Conceptually:

```text
Producer
   |
   | queue full?
   v
wait on condition_variable
   |
space available
   v
push event
```

and:

```text
Consumer
   |
   | queue empty?
   v
wait on condition_variable
   |
event available
   v
pop event
```

---

## Modern C++20 Thread Management

The pipeline will use:

```cpp
std::jthread
```

instead of manually managing raw `std::thread` objects where possible.

`std::jthread` provides:

- automatic joining
- cooperative cancellation support
- integration with `std::stop_token`

The consumer can therefore periodically check:

```cpp
stop_token.stop_requested()
```

and exit cleanly when shutdown is requested.

---

## Graceful Shutdown

Thread shutdown is an important part of a production-style asynchronous system.

A consumer may be sleeping while waiting for data:

```text
consumer
   ↓
condition_variable::wait()
```

If the program simply exits without notifying it, the thread may remain blocked.

The queue will therefore support a closed state.

Conceptually:

```text
Producer finishes
      ↓
queue.close()
      ↓
wake waiting consumers
      ↓
consumer drains remaining events
      ↓
consumer exits
```

This allows the pipeline to terminate without deadlocks or abandoned work.

---

## Expected Architecture

By the end of this notebook, the pipeline should look like:

```text
┌──────────────────────┐
│ SyntheticEventSource │
└──────────┬───────────┘
           │
           │ producer thread
           ▼
┌──────────────────────┐
│ BoundedQueue          │
│ <ProfileEvent>        │
└──────────┬───────────┘
           │
           │ consumer thread
           ▼
┌──────────────────────┐
│ Event Processing     │
│ Serializer / Sink    │
└──────────────────────┘
```

---

## Key Concepts Practiced

This notebook introduces several important systems-programming concepts:

```text
producer-consumer architecture
shared mutable state
mutex
condition variable
bounded buffering
backpressure
thread lifecycle
std::jthread
std::stop_token
graceful shutdown
```

These concepts will later be reused when the queue feeds the TCP transport layer.

---

## Completion Criteria

This notebook is complete when:

```text
[ ] BoundedQueue<T> is implemented
[ ] Queue capacity is enforced
[ ] Producer waits when the queue is full
[ ] Consumer waits when the queue is empty
[ ] Multiple threads can safely access the queue
[ ] SyntheticEventSource runs in a producer thread
[ ] Event processing runs in a consumer thread
[ ] std::jthread is used for thread lifecycle management
[ ] The queue supports graceful shutdown
[ ] All events are processed without loss or deadlock
[ ] The implementation builds successfully with CMake
```

Once this pipeline is working, the next stage will add batching and prepare the event stream for network transport.

## Standalone Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The setup installs and verifies the minimal development tools required for the thread-safe C++20 pipeline.

Required tools:

- GCC / G++ with C++20 support
- CMake
- Ninja
- Git
- POSIX thread support

This notebook does not require a GPU.

The focus is on host-side systems programming:

- multithreading
- synchronization
- bounded queues
- producer-consumer execution
- thread lifecycle management
- graceful shutdown

In [1]:
!sudo apt-get update -qq

!sudo apt-get install -y \
    build-essential \
    g++ \
    cmake \
    ninja-build \
    git

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
build-essential is already the newest version (12.10ubuntu1).
g++ is already the newest version (4:13.2.0-7ubuntu1).
g++ set to manually installed.
cmake is already the newest version (3.28.3-1build7).
git is already the newest version (1:2.43.0-1ubuntu7.3).
The following NEW packages will be installed:
  ninja-build
0 upgraded, 1 newly installed, 0 to remove and 64 not upgraded.
Need to get 129 kB of archives.
After this operation, 364 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble/universe amd64 ninja-build amd64 1.11.1-2 [129 kB]
Fetched 129 kB in 0s (323 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so

In [2]:
!echo "=== Compiler ==="
!g++ --version | head -n 1

!echo ""
!echo "=== CMake ==="
!cmake --version | head -n 1

!echo ""
!echo "=== Ninja ==="
!ninja --version

=== Compiler ===
g++ (Ubuntu 13.3.0-6ubuntu2~24.04.1) 13.3.0

=== CMake ===
cmake version 3.31.10

=== Ninja ===
1.11.1


In [3]:
!rm -rf distributed-profiler

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks

## Prerequisite Core Components

This notebook builds on the core data model and interfaces introduced in the previous notebook.

To keep the notebook fully standalone, the required source files are recreated below.

The reused components are:

```text
include/
├── profile_event.hpp
├── event_source.hpp
├── serializer.hpp
├── data_sink.hpp
├── synthetic_event_source.hpp
├── simple_text_serializer.hpp
└── console_sink.hpp

src/
├── synthetic_event_source.cpp
├── simple_text_serializer.cpp
└── console_sink.cpp

In [4]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent =
    std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [5]:
%%writefile distributed-profiler/include/event_source.hpp

#pragma once

#include "profile_event.hpp"

#include <optional>

class IEventSource {
public:
    virtual ~IEventSource() = default;

    virtual std::optional<ProfileEvent>
    next_event() = 0;
};

Writing distributed-profiler/include/event_source.hpp


In [6]:
%%writefile distributed-profiler/include/serializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <vector>

class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};

Writing distributed-profiler/include/serializer.hpp


In [7]:
%%writefile distributed-profiler/include/data_sink.hpp

#pragma once

#include "profile_event.hpp"

class IDataSink {
public:
    virtual ~IDataSink() = default;

    virtual void write(
        const ProfileEvent& event
    ) = 0;
};

Writing distributed-profiler/include/data_sink.hpp


In [8]:
%%writefile distributed-profiler/include/synthetic_event_source.hpp

#pragma once

#include "event_source.hpp"

#include <cstddef>
#include <optional>
#include <vector>

class SyntheticEventSource : public IEventSource {
public:
    SyntheticEventSource();

    std::optional<ProfileEvent>
    next_event() override;

private:
    std::vector<ProfileEvent> events_;
    std::size_t current_index_;
};

Writing distributed-profiler/include/synthetic_event_source.hpp


In [9]:
%%writefile distributed-profiler/src/synthetic_event_source.cpp

#include "synthetic_event_source.hpp"

SyntheticEventSource::SyntheticEventSource()
    : current_index_(0) {

    events_.push_back(
        KernelEvent{
            1000,
            200,
            0,
            "gemm_kernel"
        }
    );

    events_.push_back(
        MemoryEvent{
            1500,
            0,
            4096,
            "H2D"
        }
    );

    events_.push_back(
        KernelEvent{
            2000,
            150,
            0,
            "softmax_kernel"
        }
    );

    events_.push_back(
        MemoryEvent{
            2300,
            0,
            8192,
            "D2H"
        }
    );
}

std::optional<ProfileEvent>
SyntheticEventSource::next_event() {

    if (current_index_ < events_.size()) {
        return events_[current_index_++];
    }

    return std::nullopt;
}

Writing distributed-profiler/src/synthetic_event_source.cpp


In [10]:
%%writefile distributed-profiler/include/simple_text_serializer.hpp

#pragma once

#include "serializer.hpp"

class SimpleTextSerializer : public ISerializer {
public:
    std::vector<std::byte>
    serialize(
        const ProfileEvent& event
    ) const override;
};

Writing distributed-profiler/include/simple_text_serializer.hpp


In [11]:
%%writefile distributed-profiler/src/simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <string>
#include <type_traits>
#include <variant>

std::vector<std::byte>
SimpleTextSerializer::serialize(
    const ProfileEvent& event
) const {

    std::string text;

    std::visit(
        [&](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                text =
                    "KERNEL|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.duration_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    e.kernel_name;
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                text =
                    "MEMORY|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    std::to_string(e.bytes) + "|" +
                    e.operation;
            }
        },
        event
    );

    std::vector<std::byte> bytes;
    bytes.reserve(text.size());

    for (char c : text) {
        bytes.push_back(
            static_cast<std::byte>(c)
        );
    }

    return bytes;
}

Writing distributed-profiler/src/simple_text_serializer.cpp


In [12]:
%%writefile distributed-profiler/include/console_sink.hpp

#pragma once

#include "data_sink.hpp"

class ConsoleSink : public IDataSink {
public:
    void write(
        const ProfileEvent& event
    ) override;
};

Writing distributed-profiler/include/console_sink.hpp


In [13]:
%%writefile distributed-profiler/src/console_sink.cpp

#include "console_sink.hpp"

#include <iostream>
#include <type_traits>
#include <variant>

void ConsoleSink::write(
    const ProfileEvent& event
) {

    std::visit(
        [](const auto& e) {
            using T =
                std::decay_t<decltype(e)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                std::cout
                    << "[KernelEvent] "
                    << "kernel=" << e.kernel_name
                    << ", device=" << e.device_id
                    << ", timestamp=" << e.timestamp_ns
                    << ", duration=" << e.duration_ns
                    << " ns\n";
            }
            else if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                std::cout
                    << "[MemoryEvent] "
                    << "operation=" << e.operation
                    << ", device=" << e.device_id
                    << ", bytes=" << e.bytes
                    << ", timestamp=" << e.timestamp_ns
                    << " ns\n";
            }
        },
        event
    );
}

Writing distributed-profiler/src/console_sink.cpp


## Step 1 — Implement a Bounded Queue

The first step is to implement a reusable thread-safe bounded queue.

The queue will sit between the producer and consumer threads:

```text
Producer Thread
      ↓
   push()
      ↓
BoundedQueue<T>
      ↓
    pop()
      ↓
Consumer Thread
```

The queue has a fixed capacity.

If the producer tries to insert data while the queue is full, it must wait until the consumer removes an item.

If the consumer tries to remove data while the queue is empty, it must wait until the producer inserts a new item.

The initial queue will provide three main operations:

```cpp
push(T item)
pop()
close()
```

- `push()` inserts an item into the queue.
- `pop()` removes and returns the next available item.
- `close()` indicates that no additional items will be produced and allows waiting consumers to exit cleanly after all remaining items have been processed.

The queue will internally use:

```text
std::deque<T>
std::mutex
std::condition_variable
bool closed_
std::size_t capacity_
```

The main goals of this step are:

- Keep shared queue state thread-safe.
- Enforce a fixed queue capacity.
- Block producers when the queue is full.
- Block consumers when the queue is empty.
- Support graceful shutdown.
- Make the queue reusable for different data types through `template<typename T>`.

In this project, the queue will later be instantiated as:

```cpp
BoundedQueue<ProfileEvent>
```

The expected behavior is:

```text
Producer faster than consumer
        ↓
Queue becomes full
        ↓
Producer waits
        ↓
Consumer removes an event
        ↓
Space becomes available
        ↓
Producer continues
```

This bounded queue provides the first backpressure mechanism in the profiling pipeline.

In [ ]:
%%writefile distributed-profiler/include/bounded_queue.hpp

#pragma once

#include <condition_variable>
#include <cstddef>
#include <deque>
#include <mutex>
#include <optional>
#include <utility>

template <typename T>
class BoundedQueue {
public:
    explicit BoundedQueue(std::size_t capacity)
        : capacity_(capacity),
          closed_(false) {}

    bool push(T item) {
        std::unique_lock<std::mutex> lock(mutex_);

        not_full_.wait( // predidate, if not closed or there is space, mutex_ will be returned and data will come back
            lock,
            [this]() {
                return queue_.size() < capacity_ || closed_;
            }
        );

        if (closed_) {
            return false;
        }

        queue_.push_back(std::move(item));

        lock.unlock();

        not_empty_.notify_one();

        return true;
    }

    std::optional<T> pop() {
        std::unique_lock<std::mutex> lock(mutex_);

        not_empty_.wait(
            lock,
            [this]() {
                return !queue_.empty() || closed_;
            }
        );

        if (queue_.empty() && closed_) {
            return std::nullopt;
        }

        T item = std::move(queue_.front());
        queue_.pop_front();

        lock.unlock();

        not_full_.notify_one();

        return item;
    }

    void close() {
        {
            std::lock_guard<std::mutex> lock(mutex_);
            closed_ = true;
        }

        not_empty_.notify_all();
        not_full_.notify_all();
    }

private:
    std::deque<T> queue_;

    std::size_t capacity_;

    std::mutex mutex_;

    std::condition_variable not_empty_;
    std::condition_variable not_full_;

    bool closed_;
};

## Step 2 — Validate the Bounded Queue

Before connecting `BoundedQueue<T>` to the profiling pipeline, the queue should first be validated independently with a simple data type.

The goal of this step is to confirm that the queue correctly handles:

- fixed capacity
- producer blocking when the queue is full
- consumer blocking when the queue is empty
- wake-up notifications between producer and consumer
- graceful shutdown through `close()`

A simple `BoundedQueue<int>` will be used for this validation.

The test will use:

```text
capacity = 2
```

and two threads:

```text
Producer Thread
      ↓
 push(1)
 push(2)
 push(3)
      ↓
BoundedQueue<int>
      ↓
 pop()
      ↓
Consumer Thread
```

The consumer will intentionally process items more slowly than the producer.

This should cause the queue to become full:

```text
Producer pushes 1
        ↓
Producer pushes 2
        ↓
Queue is full
        ↓
Producer tries to push 3
        ↓
Producer waits
```

When the consumer removes an item:

```text
Consumer pops 1
        ↓
Queue has available space
        ↓
not_full_.notify_one()
        ↓
Producer wakes up
        ↓
Producer pushes 3
```

This verifies that the bounded queue provides backpressure correctly.

The test will also validate queue shutdown.

After the producer finishes inserting all items, it will call:

```cpp
queue.close();
```

The consumer should continue processing any items that are still buffered.

Once the queue is both:

```text
closed
AND
empty
```

`pop()` should return:

```cpp
std::nullopt
```

and the consumer thread should exit normally.

The expected shutdown sequence is:

```text
Producer finishes
        ↓
queue.close()
        ↓
Consumer wakes up
        ↓
Remaining items are consumed
        ↓
Queue becomes empty
        ↓
pop() returns std::nullopt
        ↓
Consumer exits
```

This test isolates the synchronization behavior of `BoundedQueue<T>` before introducing `ProfileEvent`, `SyntheticEventSource`, serialization, or other profiling-specific components.

In [ ]:
%%writefile distributed-profiler/tests/test_bounded_queue.cpp

#include "bounded_queue.hpp"

#include <chrono>
#include <iostream>
#include <thread>

using namespace std::chrono_literals;

int main() {
    // TODO 1:
    // Create a bounded queue with capacity = 2.
    //
    // Example:
    // BoundedQueue<int> queue(...);


    // TODO 2:
    // Create a producer thread.
    //
    // Producer behavior:
    // - push 1
    // - print "Producer pushed 1"
    // - push 2
    // - print "Producer pushed 2"
    // - push 3
    // - print "Producer pushed 3"
    // - call queue.close()
    //
    // Important:
    // push(3) should eventually block if the queue is full.


    // TODO 3:
    // Create a consumer thread.
    //
    // Consumer behavior:
    // - sleep briefly before starting
    // - repeatedly call queue.pop()
    // - if pop() returns an item:
    //       print the value
    //       sleep for a short period to simulate slow processing
    // - if pop() returns std::nullopt:
    //       exit the loop


    // TODO 4:
    // Wait for both threads to finish.
    //
    // If using std::thread:
    // producer.join();
    // consumer.join();


    std::cout << "BoundedQueue test finished\n";

    return 0;
}